## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 📝 Override Result with Middleware

## Industry Use Case: Market Data Enrichment

This notebook demonstrates how middleware can **modify or override** agent responses.

| Feature | FSI Application |
|---------|-----------------|
| **Result Transformation** | Add compliance disclaimers |
| **Result Enrichment** | Append regulatory notices |
| **Result Validation** | Ensure responses meet standards |

In [21]:
import os
from dotenv import load_dotenv

load_dotenv('../../.env', override=True)

PROJECT_ENDPOINT = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

print(f"✅ Environment loaded")

✅ Environment loaded


In [22]:
from collections.abc import AsyncIterable, Awaitable, Callable

from agent_framework import (
    Agent,
    AgentContext,
    AgentResponse,
    AgentResponseUpdate,
    Content,
    Message,
    agent_middleware,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("✅ All imports loaded")

✅ All imports loaded


## Define Market Data Tool

In [23]:
def get_stock_price(symbol: str) -> str:
    """Get current stock price for a symbol."""
    prices = {"MSFT": 425.50, "AAPL": 178.25, "GOOGL": 142.80}
    price = prices.get(symbol.upper(), 100.00)
    return f"{symbol.upper()} is currently trading at ${price:.2f}"

print("✅ Tool defined: get_stock_price")

✅ Tool defined: get_stock_price


## Compliance Disclaimer Middleware

Appends regulatory disclaimers to all market data responses.

In [24]:
@agent_middleware
async def compliance_disclaimer_middleware(
    context: AgentContext, 
    next: Callable[[], Awaitable[None]]
) -> None:
    """Appends compliance disclaimers to all responses."""
    
    # Let the agent process normally first
    await next()
    
    # Add disclaimer to the result
    if context.result is not None:
        disclaimer = "\n\n⚠️ DISCLAIMER: This information is for educational purposes only and does not constitute financial advice."
        
        if context.stream:
            # For streaming: use a transform hook to append disclaimer at end
            from agent_framework import ResponseStream

            original_result = context.result

            async def add_disclaimer_stream() -> AsyncIterable[AgentResponseUpdate]:
                async for chunk in original_result:
                    yield chunk
                # Append disclaimer at end
                yield AgentResponseUpdate(contents=[Content.from_text(disclaimer)])

            context.result = ResponseStream(add_disclaimer_stream())
        else:
            # For non-streaming: use .text property of response
            original_text = context.result.text or ""
            context.result = AgentResponse(
                messages=[Message(role="assistant", text=original_text + disclaimer)]
            )

print("✅ Compliance middleware defined")

✅ Compliance middleware defined


## Example 1: Non-streaming Result Override

In [25]:
async def run_non_streaming_example():
    """Non-streaming response with disclaimer appended."""
    print("\n--- Example 1: Non-streaming Result Override ---\n")
    
    credential = AzureCliCredential()
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=credential,
    )
    agent = Agent(
        client=client,
        name="MarketDataAgent",
        instructions="You provide stock market data. Use the tool to get prices.",
        tools=[get_stock_price],
        middleware=[compliance_disclaimer_middleware],
    )

    print("User: What's the price of MSFT?")
    result = await agent.run("What's the price of MSFT?")
    print(f"Agent: {result.text}")

await run_non_streaming_example()


--- Example 1: Non-streaming Result Override ---

User: What's the price of MSFT?
Agent: The current price of MSFT (Microsoft) is $425.50.

⚠️ DISCLAIMER: This information is for educational purposes only and does not constitute financial advice.


## Example 2: Streaming Result Override

In [26]:
async def run_streaming_example():
    """Streaming response with disclaimer appended at end."""
    print("\n--- Example 2: Streaming Result Override ---\n")
    
    credential = AzureCliCredential()
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=credential,
    )
    agent = Agent(
        client=client,
        name="MarketDataAgent",
        instructions="You provide stock market data. Use the tool to get prices.",
        tools=[get_stock_price],
        middleware=[compliance_disclaimer_middleware],
    )

    print("User: What's the price of AAPL?")
    print("Agent: ", end="", flush=True)
    async for chunk in agent.run("What's the price of AAPL?", stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print()

await run_streaming_example()


--- Example 2: Streaming Result Override ---

User: What's the price of AAPL?
Agent: The current price of AAPL (Apple Inc.) is $178.25.

⚠️ DISCLAIMER: This information is for educational purposes only and does not constitute financial advice.


## Key Takeaways

| Technique | When to Use | FSI Example |
|-----------|-------------|-------------|
| `await next(context)` first | Let agent process normally | Get original response |
| Modify `context.result` | Transform output | Add compliance disclaimers |
| `context.is_streaming` | Detect execution mode | Handle both patterns |
| Async generator | Stream overrides | Append text chunks |

## Next Steps
- **Shared State** (notebook 9) - Cross-middleware communication